In [43]:
# python
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import funciones as fn

In [44]:
# Rutas y nombre de la columna objetivo
train_path = "dataset/train.csv"
val_path = "dataset/val.csv"
target_column = "shares"

# Cargar datos
df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

In [45]:
# Columnas especificadas manualmente vía funciones
numeric_cols = fn.columnas_numericas()
#numeric_cols = fn.columnas_cantidad()
#ohe_cols = fn.columnas_one_hot()
ohe_cols = fn.columnas_one_hot_dia_semana() + fn.columnas_one_hot_tematica()
use_imputer_for_ohe = False  # True si las OHE tienen NaNs

In [46]:

# Validar existencia de columnas
missing = [c for c in numeric_cols + ohe_cols + [target_column] if c not in df_train.columns]
if missing:
    raise ValueError(f"Faltan columnas en `train`: {missing}")
missing = [c for c in numeric_cols + ohe_cols + [target_column] if c not in df_val.columns]
if missing:
    raise ValueError(f"Faltan columnas en `val`: {missing}")

In [47]:

# Separar X / y
X_train = df_train.drop(columns=[target_column])
y_train = df_train[target_column]
X_val = df_val.drop(columns=[target_column])
y_val = df_val[target_column]

In [48]:
# Construir transformadores
transformers = []
if numeric_cols:
    num_pipeline = Pipeline([('scaler', StandardScaler())])
    transformers.append(('num', num_pipeline, numeric_cols))

if ohe_cols:
    transformers.append(('ohe_given', 'passthrough', ohe_cols))

if not transformers:
    raise ValueError('No se definieron columnas numéricas ni OHE.')

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

# Modelo Random Forest con parámetros fijos
rf = RandomForestRegressor(
    n_estimators=2000,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=4,
    max_features='log2',
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

# Pipeline con el preprocesamiento + modelo
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', rf)
])

# Entrenamiento directo
pipeline.fit(X_train, y_train)

# El modelo entrenado es el pipeline
best_model = pipeline

# Función de evaluación usando el mejor modelo
def evaluar(nombre, X, y_true):
    y_pred = best_model.predict(X)  # revertimos log1p
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"--- {nombre} ---")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R2: {r2:.4f}")

# Evaluar en validación
evaluar('validation', X_val, y_val)

--- validation ---
MSE: 1043828.6315
RMSE: 1021.6793
MAE: 771.2393
R2: 0.1160


In [49]:
# python
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score
# instala si hace falta: xgboost, lightgbm, catboost
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

models = {
    "SklearnGB": GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
    "HistGB": HistGradientBoostingRegressor(max_iter=200, max_depth=10, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, objective='reg:squarederror', random_state=42, n_jobs=-1),
    "LightGBM": lgb.LGBMRegressor(n_estimators=200, max_depth=-1, learning_rate=0.05, random_state=42, n_jobs=-1),
    "CatBoost": CatBoostRegressor(iterations=500, depth=6, learning_rate=0.05, random_state=42, verbose=0)
}

results = {}
for name, est in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('reg', est)])  # usa el preprocessor ya definido en tu notebook
    # Opción A: entrenar con y en escala original
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    # Opción B (si entrenás con log1p):
    # pipe.fit(X_train, np.log1p(y_train))
    # preds = np.expm1(pipe.predict(X_val))
    results[name] = r2_score(y_val, preds)
    print(f"{name}: R2 = {results[name]:.4f}")

SklearnGB: R2 = 0.1028
HistGB: R2 = 0.1051
XGBoost: R2 = 0.1122
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006171 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5817
[LightGBM] [Info] Number of data points in the train set: 25901, number of used features: 46
[LightGBM] [Info] Start training from score 1660.559554


C:\Users\JP_La\Repositorios\Facultad\2025\CDD\CienciaDeDatos_2025\conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM: R2 = 0.1152
CatBoost: R2 = 0.1150
